In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")

In [4]:
df_ground_truth.head()

,question,document
0,Is it possible to enroll in the course even if...,74eb249bbf
1,What are the requirements for earning a course...,74eb249bbf
2,Is there a specific deadline for submitting th...,74eb249bbf
3,"If I join the course late, can I still qualify...",74eb249bbf
4,Are there any time constraints for project sub...,74eb249bbf


In [5]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [6]:
ground_truth[10]

{'question': 'Will students get a Zoom link to join the live office hours?',
 'document': '489dd1c9d9'}

In [7]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [8]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': 'db78580409

In [9]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [10]:
q = ground_truth[0]
q

{'question': "Is it possible to enroll in the course even if I've just found it?",
 'document': '74eb249bbf'}

In [11]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [12]:
results = text_search(q['question'])
results

[{'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '610ccb23c0',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'What happens to code s

In [13]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

0fab61eca2 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
610ccb23c0 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [14]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 1, 0, 0, 0]

In [15]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [16]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it possible to enroll in the course even if I've just found it?


[0, 1, 0, 0, 0]

In [18]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

Where can I watch the live workshop sessions?


[1, 0, 0, 0, 0]

In [19]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 0, 0, 0]

Where can I access the main platform for the LLM Zoomcamp course?


[1, 0, 0, 0, 0]

In [20]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [21]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/575 [00:00<?, ?it/s]

In [22]:
relevance[:15]

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [23]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [24]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [25]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/575 [00:00<?, ?it/s]

In [26]:
sample = relevance_total[:15]

In [27]:
sample

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [28]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.8

In [29]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [30]:
hit_rate(relevance)

0.8469565217391304

In [31]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6055555555555556

In [32]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [33]:
mrr(sample)

0.6055555555555556

In [34]:
mrr(relevance)

0.702550724637681

In [35]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [36]:
evaluate(ground_truth, text_search)

  0%|          | 0/575 [00:00<?, ?it/s]

{'hit_rate': 0.8469565217391304, 'mrr': 0.702550724637681}

In [37]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [38]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/575 [00:00<?, ?it/s]

{'hit_rate': 0.8643478260869565, 'mrr': 0.7330144927536231}

In [39]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [40]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/575 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8904347826086957, 'mrr': 0.7798550724637677}


  0%|          | 0/575 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8921739130434783, 'mrr': 0.778260869565217}


  0%|          | 0/575 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8469565217391304, 'mrr': 0.702550724637681}


  0%|          | 0/575 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8017391304347826, 'mrr': 0.6615652173913044}


  0%|          | 0/575 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7669565217391304, 'mrr': 0.6316811594202899}


In [41]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [42]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/575 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/575 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/575 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/575 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/575 [00:00<?, ?it/s]

In [43]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.960000,0.852928
34,5.0,10.0,0.2,0.953043,0.852000
18,2.0,4.0,0.1,0.953043,0.851797
6,1.0,4.0,0.1,0.958261,0.850725
33,5.0,10.0,0.1,0.951304,0.850696
19,2.0,4.0,0.2,0.958261,0.850522
35,5.0,10.0,0.5,0.958261,0.850522
3,1.0,2.0,0.1,0.958261,0.850522
20,2.0,4.0,0.5,0.951304,0.850464
4,1.0,2.0,0.2,0.956522,0.850348


In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)